### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [5]:
## step1 : Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)


In [6]:
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [7]:
### step 2: Vector Store
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks,embedding_model)

## step 3:MMR Retriever
retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":5})
retriever

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 304.40it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000249B290FE00>, search_type='mmr', search_kwargs={'k': 5})

In [9]:
## step 4 : LLM and Prompt

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm=init_chat_model("groq:openai/gpt-oss-120b") 
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000249B441B0E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000249B2A04830>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")

query_expansion_chain=query_expansion_prompt| llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000249B441B0E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000249B2A04830>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOutputParser()

In [11]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'**Expanded query**\n\n```\n(\n  "LangChain memory"\n  OR "LangChain conversation memory"\n  OR "LangChain state management"\n  OR "LangChain context handling"\n  OR "LangChain session storage"\n  OR "LangChain memory buffer"\n  OR "LangChain memory module"\n  OR "LangChain memory component"\n  OR "LangChain memory API"\n  OR "LangChain memory implementation"\n  OR "LangChain memory design patterns"\n  OR "LangChain memory providers"\n  OR "LangChain persistent memory"\n  OR "LangChain short‑term memory"\n  OR "LangChain long‑term memory"\n  OR "LangChain memory for agents"\n  OR "LangChain memory for chatbots"\n  OR "LangChain memory for retrieval‑augmented generation"\n  OR "LangChain memory for chain‑of‑thought"\n  OR "LangChain memory types"\n  OR "LangChain memory buffers"\n  OR "LangChain memory classes"\n  OR "LangChain memory interfaces"\n  OR "LangChain memory stores"\n  OR "LangChain memory handling"\n  OR "LangChain memory examples"\n  OR "LangChain memory tutorial"\n  OR "L

In [12]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

In [13]:
# Step 5: Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    | document_chain
)

In [14]:
# Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

**Expanded query**

```
{
  "input": "What kinds of memory mechanisms, state‑management techniques, and context‑storage options does LangChain provide? 
  Include details on the various Memory classes and back‑ends such as ConversationBufferMemory, 
  ConversationSummaryMemory, ConversationBufferWindowMemory, ConversationSummaryBufferMemory, 
  ConversationEntityMemory, VectorStoreMemory (FAISS, Chroma, Milvus, Pinecone, etc.), 
  RedisCacheMemory, SQLMemory, PickleMemory, InMemoryMemory, PersistentMemory, 
  RetrievalMemory, and any other supported memory abstractions or plugins. 
  Also ask for synonyms like ‘state retention’, ‘contextual memory’, ‘session memory’, 
  ‘memory back‑ends’, ‘memory stores’, and related technical terms (e.g., “memory interface”, 
  “memory provider”, “memory layer”, “memory stack”, “agent memory”)."
}
```
✅ Answer:
 LangChain provides built‑in memory modules such as:

- **ConversationBufferMemory** – stores the full dialogue history so the LLM can see ev

In [15]:
# Step 6: Run query
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

**Expanded query**

```json
{
  "input": "CrewAI agents OR \"Crew AI\" OR autonomous AI agents OR multi‑agent systems OR collaborative AI agents OR LLM‑powered agents OR AI agent orchestration OR agentic AI frameworks OR crew scheduling optimization OR AI‑driven crew coordination OR CrewAI platform documentation OR crew.ai OR CrewAI GitHub repository OR CrewAI use cases OR implementation examples OR technical architecture OR prompt engineering for CrewAI agents"
}
```
✅ Answer:
 **CrewAI agents are autonomous “team‑members” that work together in a structured workflow.**  

- **Defined roles** – Each agent is given a specific purpose such as *researcher*, *planner*, *executor*, etc. The role determines what the agent focuses on and which tools it can use.  
- **Semi‑independent operation** – Within the crew, agents run their own reasoning loops, maintain their own short‑term memory, and make decisions, but they do so under the coordination of the CrewAI orchestrator.  
- **Collaborative